## Cost Basis Analysis

This notebook defines the **economic cost-basis model** used by Fidelity Tracker.

### Scope

The analysis is performed per `account_number` and `symbol`.

Tax accounting is intentionally out of scope:

* no wash-sale adjustments
* no tax-lot reporting
* no short-term / long-term classification
* no estimated tax liability

The tracker uses **weighted-average cost allocation** because the product is focused on ticker-level and portfolio-level analytics rather than individual tax lots.

### Core state

For each account and symbol, only three values need to be carried forward:

* `remaining_quantity`
* `remaining_cost_basis`
* `realized_profit`

`average_cost` is derived: remaining_cost_basis/remaining_quantity

`amount` is treated as the authoritative cash value:

* purchases / reinvestments: negative `amount`
* sales: positive `amount`

`price × quantity` is not used as the authoritative transaction value because Fidelity's displayed price can differ slightly from the final cash amount.

When a position is fully closed:
* remaining quantity = 0
* remaining cost basis = 0

and the realized Profit for the position must reconcile to the signed cash flow:
* realized Profit = sum(amount)

for the position-changing transactions included in this cost-basis calculation.


In [18]:
from pathlib import Path
import sys

# Create PROJECT_ROOT and add it to sys.path so Python can search for customized module
CURRENT_DIR = Path.cwd()
PROJECT_ROOT = CURRENT_DIR.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [19]:
import pandas as pd
import numpy as np
from backend.pipeline.preprocessing import preprocessing

In [20]:
data = preprocessing(verbose=False)

2026-08-16 16:36:04 | WARNING  | Repeated transaction keys found: 3


### Weighted-average allocation rules

The service does not need to maintain a separate "remaining dollar amount" state. The dollar amount still attached to an open position is represented by `remaining_cost_basis`.

For each symbol, transactions are processed by date.

If both buys and sells happen on the same day, all buys are processed first. The sells are then processed using the updated weighted-average cost.

At the end of each day:

`remaining_quantity >= 0`

If the quantity becomes negative, the transaction history or processing logic is invalid.

#### Buy / reinvestment

For a purchase:

`purchase_cost = -amount`

Since Fidelity records purchases as negative amounts, multiplying by -1 gives the actual purchase cost.

Update the position:

`remaining_quantity = remaining_quantity + quantity_bought`

`remaining_cost_basis = remaining_cost_basis + purchase_cost`

Then calculate the new average cost:

`average_cost = remaining_cost_basis / remaining_quantity`

A buy can change the average cost because both the remaining quantity and remaining cost basis increase.

#### Sell

Before processing a sale, calculate the current average cost:

`average_cost = remaining_cost_basis / remaining_quantity`

Calculate the cost basis assigned to the shares being sold:

`basis_removed = average_cost * quantity_sold`

Calculate the realized Profit from the sale:

`realized_profit_from_sale = sale_amount - basis_removed`

Update the position:

`remaining_quantity = remaining_quantity - quantity_sold`

`remaining_cost_basis = remaining_cost_basis - basis_removed`

`realized_profit = realized_profit + realized_profit_from_sale`

A sell does not change the average cost of the shares that remain. It only removes quantity and cost basis using the current weighted-average cost.

If the remaining quantity reaches zero, set both:

`remaining_quantity = 0`

`remaining_cost_basis = 0`

This prevents small floating-point leftovers after a position is fully closed.

#### Corporate Action

Corporate actions such as stock split and reverse stock will change the total quantity by a multiplier. 

The `remaining_cost_basis` stays the same, but it's important to calculate an updated `remaining_quantity` before `realized_profit`


In [21]:
# reinvestment of mutual fund are in the form of shares and has to be considered
trade_data = data[
    data["transaction_type"].eq("trade")
    | data["transaction_subtype"].eq("reinvestment")
].drop_duplicates().reset_index(drop=True)

In [22]:
trade_data.shape

(995, 21)

In [23]:
trade_data.columns

Index(['run_date', 'account', 'account_number', 'action', 'symbol',
       'description', 'type', 'price', 'quantity', 'commission', 'fees',
       'accrued_interest', 'amount', 'settlement_date', 'transaction_type',
       'transaction_subtype', 'cusip', 'security_name', 'security_type',
       'security_type_raw', 'security_source'],
      dtype='str')

In [24]:
trade_data['transaction_subtype'].unique()

<StringArray>
['bought', 'sold', 'reinvestment']
Length: 3, dtype: str

In [25]:
def populate_symbol_statistic(transaction_data: pd.DataFrame) -> dict:
    '''
    For each account and symbol, calculate the remaining quantity, remaining cost basis, and realized gain/loss
    '''
    priority = {
        "reinvestment": 0,
        "bought": 1,
        "sold": 2,
    }

    transaction_data["transaction_priority"] = (
        transaction_data["transaction_subtype"].map(priority)
    )

    transaction_data.sort_values(
        by=["run_date", "transaction_priority"],
        inplace=True,
    )
    
    # account: {name: {remaining_quantity, remaining_cost_basis, realized_gain_loss:}}
    symbol_statistic = {} 
    unique_symbols = transaction_data['symbol'].unique()
    unique_accounts = transaction_data['account_number'].unique()

    for account in unique_accounts:
        # Create account entry
        if account not in symbol_statistic:
            symbol_statistic[account] = {}
        
        for symbol in unique_symbols:
            transactions = transaction_data[
                (transaction_data['symbol'] == symbol) & (transaction_data['account_number'] == account)
            ]

            if transactions.empty:
                continue # Skip if there are no transactions for this symbol and account

            # Create symbol entry
            if symbol not in symbol_statistic[account]:
                symbol_statistic[account][symbol] = {
                    'remaining_quantity': 0.0,
                    'remaining_cost_basis': 0.0,
                    'realized_gain_loss': 0.0
                }

            for transaction in transactions.itertuples():
                quantity = transaction.quantity
                amount = transaction.amount
                action = transaction.transaction_subtype

                # Reinvestment 
                if action == 'reinvestment':
                    if quantity < -1e-9:
                        raise ValueError(f"Reinvestment transaction has negative quantity: {quantity}")

                    acquisition_cost = -amount 
                    symbol_statistic[account][symbol]['remaining_quantity'] += quantity
                    symbol_statistic[account][symbol]['remaining_cost_basis'] += acquisition_cost

                # Bought
                elif action == 'bought':
                    if quantity < -1e-9:
                        raise ValueError(f"Buy transaction has negative quantity: {quantity}")

                    purchase_cost = -amount 
                    symbol_statistic[account][symbol]['remaining_quantity'] += quantity
                    symbol_statistic[account][symbol]['remaining_cost_basis'] += purchase_cost

                # Sold
                elif action == 'sold':
                    if quantity > 1e-9:
                        raise ValueError(f"Sell transaction has positive quantity: {quantity}")
                    
                    quantity_sold = -quantity
                    average_cost = symbol_statistic[account][symbol]['remaining_cost_basis'] / symbol_statistic[account][symbol]['remaining_quantity'] if symbol_statistic[account][symbol]['remaining_quantity'] > 0 else 0
                    basis_removed = average_cost * quantity_sold
                    sale_proceeds = amount # Fidelity Sold is positive
                    gain_loss = sale_proceeds - basis_removed

                    symbol_statistic[account][symbol]['remaining_quantity'] -= quantity_sold
                    symbol_statistic[account][symbol]['remaining_cost_basis'] -= basis_removed
                    symbol_statistic[account][symbol]['realized_gain_loss'] += gain_loss

                if symbol_statistic[account][symbol]['remaining_quantity'] < -1e-9:
                    raise ValueError(f"Remaining quantity for {symbol} in account {account} is negative: {symbol_statistic[account][symbol]['remaining_quantity']}")
                
                # Set remaining cost basis to zero if remaining quantity is close to zero
                if symbol_statistic[account][symbol]['remaining_quantity'] < 1e-9:
                    symbol_statistic[account][symbol]['remaining_quantity'] = 0.0
                    symbol_statistic[account][symbol]['remaining_cost_basis'] = 0.0

    # Round for reability
    for account in symbol_statistic.values():
        for stats in account.values():
            for key in stats:
                stats[key] = round(stats[key], 2)

    return symbol_statistic

In [26]:
symbol_statistic = populate_symbol_statistic(trade_data)

In [27]:
symbol_statistic.items()

dict_items([('Z31791998', {'FSPGX': {'remaining_quantity': 0.0, 'remaining_cost_basis': 0.0, 'realized_gain_loss': 54.16}, 'SOXL': {'remaining_quantity': 0.0, 'remaining_cost_basis': 0.0, 'realized_gain_loss': 254.79}, 'TSLZ': {'remaining_quantity': 0.0, 'remaining_cost_basis': 0.0, 'realized_gain_loss': -7.09}, 'PEP': {'remaining_quantity': 0.0, 'remaining_cost_basis': 0.0, 'realized_gain_loss': -2.45}, 'AVTR': {'remaining_quantity': 0.0, 'remaining_cost_basis': 0.0, 'realized_gain_loss': -0.15}, 'VLRS': {'remaining_quantity': 0.0, 'remaining_cost_basis': 0.0, 'realized_gain_loss': -22.69}, 'SCHD': {'remaining_quantity': 0.0, 'remaining_cost_basis': 0.0, 'realized_gain_loss': 0.19}, 'WLDS': {'remaining_quantity': 0.0, 'remaining_cost_basis': 0.0, 'realized_gain_loss': -5.57}, 'SPOT': {'remaining_quantity': 0.0, 'remaining_cost_basis': 0.0, 'realized_gain_loss': 121.02}, 'SPAXX': {'remaining_quantity': 66.05, 'remaining_cost_basis': 66.05, 'realized_gain_loss': 0.0}, 'SNAP': {'remainin

In [28]:
# Validation: For all closed positions, the realized gain/loss should equal to the sum of amount
for account, symbols in symbol_statistic.items():
    for symbol, stats in symbols.items():
        if stats['remaining_quantity'] == 0:
            transactions = trade_data[
                (trade_data['symbol'] == symbol) & (trade_data['account_number'] == account)
            ]
            total_amount = transactions['amount'].sum()
            realized_gain_loss = stats['realized_gain_loss']
            if not np.isclose(total_amount, realized_gain_loss):
                raise ValueError(f"Validation failed for {symbol} in account {account}: total amount {total_amount} does not match realized gain/loss {realized_gain_loss}")

In [29]:
for account, symbols in symbol_statistic.items():
    for symbol, stats in symbols.items():
        if stats['remaining_quantity'] > 0:
            print(f"Account: {account}, Symbol: {symbol}, Remaining Quantity: {stats['remaining_quantity']}, Remaining Cost Basis: {stats['remaining_cost_basis']}, Realized Gain/Loss: {stats['realized_gain_loss']}")

Account: Z31791998, Symbol: SPAXX, Remaining Quantity: 66.05, Remaining Cost Basis: 66.05, Realized Gain/Loss: 0.0
Account: Z31791998, Symbol: NVDA, Remaining Quantity: 5.0, Remaining Cost Basis: 1022.56, Realized Gain/Loss: 415.15
Account: Z31791998, Symbol: MU, Remaining Quantity: 4.0, Remaining Cost Basis: 3740.36, Realized Gain/Loss: -145.05
Account: Z31791998, Symbol: CRWD, Remaining Quantity: 1.0, Remaining Cost Basis: 478.08, Realized Gain/Loss: 23.91
Account: Z31791998, Symbol: SOFI, Remaining Quantity: 40.0, Remaining Cost Basis: 669.36, Realized Gain/Loss: -214.55
Account: Z31791998, Symbol: DRAM, Remaining Quantity: 20.0, Remaining Cost Basis: 1445.16, Realized Gain/Loss: 282.65
Account: Z31791998, Symbol: NOW, Remaining Quantity: 10.0, Remaining Cost Basis: 990.88, Realized Gain/Loss: -79.04
Account: Z31791998, Symbol: NKE, Remaining Quantity: 10.0, Remaining Cost Basis: 409.7, Realized Gain/Loss: 0.0


In [30]:
data[data['transaction_type'] == 'distribution']['transaction_subtype'].unique()

<StringArray>
['reinvestment', 'ordinary_dividend', 'long_term_cap_gain']
Length: 3, dtype: str

In [31]:
data['transaction_type'].unique()

<StringArray>
['transfer', 'trade', 'distribution', 'expense']
Length: 4, dtype: str

In [32]:
data.columns

Index(['run_date', 'account', 'account_number', 'action', 'symbol',
       'description', 'type', 'price', 'quantity', 'commission', 'fees',
       'accrued_interest', 'amount', 'settlement_date', 'transaction_type',
       'transaction_subtype', 'cusip', 'security_name', 'security_type',
       'security_type_raw', 'security_source'],
      dtype='str')

In [33]:
data.head()

,run_date,account,account_number,action,symbol,description,type,price,quantity,commission,...,accrued_interest,amount,settlement_date,transaction_type,transaction_subtype,cusip,security_name,security_type,security_type_raw,security_source
392,2025-03-24,Individual,Z31791998,Electronic Funds Transfer Received (Cash),NaN,No Description,Cash,NaN,0.000,NaN,...,NaN,200.00,NaT,transfer,electronic_funds_transfer,<NA>,NaN,NaN,NaN,NaN
391,2025-03-24,Individual,Z31791998,YOU BOUGHT PROSPECTUS UNDER SEPARATE COVER FID...,FSPGX,FIDELITY LARGE CAP GROWTH INDEX FUND,Cash,36.99,5.407,NaN,...,NaN,-200.00,2025-03-25,trade,bought,<NA>,FIDELITY L/C GR INDX,mutual_fund,Mutual Fund,openfigi
389,2025-04-21,Individual,Z31791998,YOU BOUGHT DIREXION SHARES ETF TRUST DAILY SEM...,SOXL,DIREXION SHARES ETF TRUST DAILY SEMICONDUCTOR ...,Cash,8.42,0.767,NaN,...,NaN,-6.46,2025-04-22,trade,bought,<NA>,DIREX DAIL SEMI BU 3X ET-USD,mutual_fund,Mutual Fund,openfigi
388,2025-04-21,Individual,Z31791998,YOU BOUGHT DIREXION SHARES ETF TRUST DAILY SEM...,SOXL,DIREXION SHARES ETF TRUST DAILY SEMICONDUCTOR ...,Cash,8.41,23.000,NaN,...,NaN,-193.54,2025-04-22,trade,bought,<NA>,DIREX DAIL SEMI BU 3X ET-USD,mutual_fund,Mutual Fund,openfigi
390,2025-04-21,Individual,Z31791998,Electronic Funds Transfer Received (Cash),NaN,No Description,Cash,NaN,0.000,NaN,...,NaN,200.00,NaT,transfer,electronic_funds_transfer,<NA>,NaN,NaN,NaN,NaN


In [34]:
data['transaction_type'].unique()

<StringArray>
['transfer', 'trade', 'distribution', 'expense']
Length: 4, dtype: str

In [37]:
data[data['transaction_type'] == 'transfer']['amount'].sum()

np.float64(8000.0)